<a href="https://colab.research.google.com/github/kyungjunoh1/LLM-workspace/blob/main/4_LangChain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### GPU 실행

In [ ]:
!pip install transformers==4.56.1 -qqq
!pip install pandas==2.2.2 -qqq
!pip install sentencepiece==0.2.1 -qqq
!pip install accelerate==1.11.0 -qqq
!pip install faiss-cpu==1.12.0 -qqq
!pip install bitsandbytes==0.47.0 -qqq
# 여기까지 설치 후 세션 다시 시작 해야 함
# langchain 설치 시 코랩 기본 설치 되어 있는 버전과 충돌 발생. pip 설치시 에러.

In [ ]:
!pip install langchain==0.3.27 -qqq
!pip install langchain-core==0.3.79 -qqq
!pip install langchain-community==0.3.31 -qqq
!pip install langchain-huggingface==0.3.1 -qqq
# 랭체인 설치시 자동으로 최신 버전의 requests==2.32.5가 깔려 경고 발생. 신경 안써도 된다

# LangChain
### LangChain
- 대규모 언어 모델(LLM)을 외부 데이터 소스와 연결하여 처리할 수 있게 해주는 프레임워크
### 속도와 효율
- 파인튜닝(Fine-tuning)은 모델 자체를 학습시키기 때문에 처리 속도가 느리고 비용이 높음
- LangChain을 활용하면 외부 데이터를 연결하여 모델이 학습 없이 참조할 수 있기 때문에 처리 속도가 빠르고 효율적
### 유연성
- 외부 데이터를 교체하거나 업데이트하는 것만으로도 모델의 답변을 빠르게 바꿀 수 있다
- 즉, 다양한 데이터를 실시간으로 활용 가능하며, RAG(검색 기반 생성) 방식과 잘 맞는다

In [ ]:
import transformers
import langchain
import langchain_community
import pandas
import sentencepiece
import accelerate
import langchain_core
import warnings
import faiss
import bitsandbytes
warnings.filterwarnings('ignore')
print(f"transformers : {transformers.__version__}" ) #4.56.1
print(f"langchain : {langchain.__version__}" ) #0.3.27
print(f"langchain-community : {langchain_community.__version__}" ) #0.3.31
print(f"pandas : {pandas.__version__}" ) #2.2.2
print(f"sentencepiece : {sentencepiece.__version__}" ) #0.2.1
print(f"accelerate : {accelerate.__version__}" ) #1.11.0
print(f"langchain-core : {langchain_core.__version__}" ) #0.3.79
print(f"faiss-cpu : {faiss.__version__}" ) #1.12.0
print(f"bitsandbytes : {bitsandbytes.__version__}" ) #0.47.0

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_community.document_loaders import CSVLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter #긴 문서를 나눠주는 기능
from langchain_core.prompts.prompt import PromptTemplate
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains import LLMChain
from langchain_huggingface import HuggingFacePipeline
import pandas as pd
from langchain.schema import Document

In [ ]:
model_id = "kakaocorp/kanana-nano-2.1b-base"
tokenizer = AutoTokenizer.from_pretrained( model_id )
model = AutoModelForCausalLM.from_pretrained( model_id , device_map="auto" ) # gpu에 모델 설정

In [ ]:
# 추론
def generate_answer(prompt):
  inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
  outputs = model.generate(
      **inputs,
      max_new_tokens=100,
      do_sample=True,
      top_p=0.9,
      temperature=0.4
  )

  answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
  return answer #, outputs

In [ ]:
generate_answer("전원 설정하는 방법 알려줘")

### 랭체인 설정
- pipeline 설정
  - 모델과, 토크나이저 등을 설정한 하나의 흐름
- HuggingFacePipeline
  - pipline의 흐름을 한번 더 포장해서 LangChain으로
  사용 할 수 있게 만드는 기능

In [ ]:
pipe = pipeline(
    "text-generation", #텍스트 생성용 모델
    model=model, # 사전 로드한 모델
    tokenizer=tokenizer, # 사용자 입력값 토큰화 기능
    max_new_tokens=300, # 한번에 생성할 최대 토큰 수
    temperature=0.4, # 생성 텍스트의 창의성(1이면 다양하게, 0이면 정확하게)
    repetition_penalty=1.2 #반복 억제
    #아래의 답변 생성 시 1.2로 진행하면 결과값이 두 번 정도 반복할 수 있다
)

In [ ]:
llm = HuggingFacePipeline(pipeline = pipe)

In [ ]:
#위에서 만든 모델에 입력할 내용 template
#모델에 지시할 내용( 동적으로 채워짐 )
# context : 학습 시킬 문서 내용
# question : 사용자 질문
template = """
너는 냉장고 제품 전문가야.
아래 메뉴얼 내용을 참고해서 질문에 답변해.
----------------
메뉴얼
{context}
----------------
사용자 질문: {question}
문서 내용만을 바탕으로 한국어로 정확하게 답변하세요.
"""

In [ ]:
prompt = PromptTemplate(template=template, input_variables=['context', 'question'])

In [ ]:
qa_chain = LLMChain(llm=llm, prompt=prompt, verbose=True)

In [ ]:
text = '''
1. 전원 및 기본 설정
- 냉장고 전원은 제품 후면 전원 코드를 연결하여 켭니다.
- 전원이 켜지면 기본 냉장 온도는 3°C, 냉동 온도는 -18°C로 설정됩니다.

2. 온도 조절 방법
- 전면 디스플레이의 '냉장' 버튼을 눌러 온도를 조절할 수 있습니다.
- 온도 범위는 0°C ~ 5°C까지 설정 가능합니다.
- '냉동' 버튼을 눌러 -15°C ~ -23°C까지 조절할 수 있습니다.

3. 급속 냉각 기능
- '급속 냉각' 버튼을 누르면 냉장실 온도가 빠르게 낮아집니다.
- 음료를 빠르게 차갑게 할 때 유용합니다.

4. 문 열림 경고
- 냉장고 문이 1분 이상 열려 있으면 경고음이 울립니다.

5. 청소 방법
- 전원을 끄고 내부를 부드러운 천으로 닦아주세요.
- 물기 제거 후 다시 전원을 켜주세요.

6. 주의사항
- 뜨거운 음식을 바로 넣지 마세요.
- 통풍구를 막지 않도록 주의하세요.
'''

In [ ]:
question = input("질문 입력")

In [ ]:
answer = qa_chain.invoke({"context":text, "question":question})

In [ ]:
answer

In [ ]:
answer['text']

### ChatPrompt

In [ ]:
prompt = ChatPromptTemplate([
    ("system","""
        너는 냉장고 제품 전문가야.
        아래 메뉴얼 내용을 참고해서 질문에 답변해.
        {context}
    """),
    ("user","{question}")
])

In [ ]:
qa_chain = LLMChain(llm=llm, prompt=prompt, verbose=True)

In [ ]:
question = input("질문 입력")
answer = qa_chain.invoke({"context":text, "question":question})

In [ ]:
answer

### RAG(검색 증강 생성)
- 모델을 새로 학습시키지 않고, 외부 지식 베이스를 검색해서 답변을 강화하는 방법
- 모델은 원래 학습된 지식 + 검색된 문서 를 기반으로 답변 생성
- RAG를 이용하면 환각 현상을 크게 줄일 수 있다
### 예시
- LangChain, LlamaIndex 등이 사용하는 접근법
- 기업 내부 문서 Q&A, 지식 기반 챗봇 등에 많이 사용됨

### 벡터 DB 활용
- 일반 DB와 비슷하지만, 숫자 벡터를 저장하고 검색하는데 최적화된 DB
- 일반 DB는 검색 키워드가 일치해야 하지만, 벡터DB는 비슷한(유사도 기반) 의미 문장을 찾아준다
- 대규모 데이터 처리 시 속도가 매우 빠르다
- 벡터 DB 순서
  - Document객체 생성 -> 작은 단위로 토큰화(Chunking) -> 임베딩(텍스트를 숫자 벡터로 변환) -> 데이터 저장
---
### Pandas VS Document
- Pandas
  - 데이터를 표 형태로 다루는 도구
  - 즉, 표형태를 이용해 데이터 분석, 처리 중심
  - 행/열, 컬럼 추가/삭제 등이 용이
  - LLM 활용 하기 위해서는 데이터 형식 변환 필요(Document)
- Document
  - LLM 용으로 바로 사용 가능
  - 단일 문서 객체 단위
    - page_content + metadata를 가지고 있다
    - 실제 텍스트 내용 + 추가정보(출처, 행 번호 등)
    - page_content, metadata 값은 실제 LLM에서 사용하는 값이기 때문에 이름은 고정으로 사용
  - Pandas처럼 표 형태로 쉽게 조작하는 기능은 없다
#### 결론
- 보통 두가지 방식의 조합으로 많이 사용하게 된다
- CSV -> Pandas -> 필요한 전처리 -> Document 변환 -> 벡터 DB

In [ ]:
import pandas as pd
data = [
    {"title": "AI 기술 발전", "content": "카카오는 인공지능 기술을 활용해 새로운 언어 모델을 발표했다. "
                                       "이 모델은 한국어 자연어 처리 성능을 대폭 향상시켰으며, "
                                       "다양한 기업 서비스에 적용될 예정이다."},
    {"title": "스마트시티 프로젝트", "content": "세종시는 스마트시티 실증단지를 통해 자율주행차, "
                                              "스마트 교통, 에너지 관리 시스템을 구축 중이다. "
                                              "이번 프로젝트에는 국내 대기업과 스타트업이 함께 참여하고 있다."},
    {"title": "친환경 에너지 전환", "content": "정부는 2035년까지 탄소 중립을 달성하기 위해 재생에너지 비율을 40%까지 높일 계획이다. "
                                               "태양광과 풍력 중심의 발전소 확대가 핵심 전략으로 제시되었다."}
]

In [ ]:
df = pd.DataFrame(data)
df

In [ ]:
Document(
    page_content=df.iloc[0]["content"],
    metadata={"title": df.iloc[0]["title"]}
)

In [ ]:
!ls

In [ ]:
df .to_csv("./sample_data.csv", index=False, encoding="utf-8-sig")

In [ ]:
!ls

In [ ]:
loader = CSVLoader(file_path="./sample_data.csv", encoding="utf-8")

In [ ]:
document = loader.load()
len(document)

### 테스트 작은 단위로 분할
- RecursiveCharacterTextSplitter을 활용하여 긴 텍스트를 잘라 더 많은 List(Document) 생성
  - LLM적용 시 텍스트가 길면 잘리거나 오류가 생길 수 있다
  - chunk_size : 문서의 길이를 보고 분할
  - chunk_overlap : 20 ~ 50% 정도 겹치게 설정
    - 문맥 연결 유지를 위해 겹친다
- RAG나 임베딩 목정
  - 벡터DB에 넣을 때는 작은 단위 청크로 나누는게 좋다
    - 검색 시 정확도가 높아진다 : 작은 단위일수록 질문과 매칭이 잘 된다
    - 임베딩 생성이 효율적 : 너무 길면 벡터 품질 저하

In [ ]:
document[0]

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=10, chunk_overlap=5)
docs = splitter.split_documents( document )
docs

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=50)
docs = splitter.split_documents(document)

### 벡터DB에 저장
- 벡터 DB에 데이터 저장 시 숫자 형식의 벡터(임베딩)로 저장해야 한다
- 한국어 임베딩 모델
  - KoSimCSE 의 모델 사용

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS

In [ ]:
# 임베딩 모델 로딩. BM-K/KoSimCSE-roberta - 한국어 최적화 모델
embeddings = HuggingFaceEmbeddings(model_name="BM-K/KoSimCSE-roberta")

In [ ]:
texts = ["임베딩 확인"]
print(embeddings.embed_documents(texts))

In [ ]:
vectorstore = FAISS.from_documents(docs, embeddings)

In [ ]:
texts = "카카오는 인공지능 기술을 활용해 새로운"
vectorstore.similarity_search(texts, k=2)

In [ ]:
data

In [ ]:
prompt = ChatPromptTemplate([
    ("system","""
        한국어 문서를 분석하는 AI agent야.
        아래 문서 내용을 보고 답해
        {context}
    """),
     ("user","{question}")
    ])
qa_chain = LLMChain(llm = llm, prompt=prompt, verbose=True)

In [ ]:
docs_result = vectorstore.similarity_search(texts, k=2)
'''
content_text[]
for doc in docs_result:
  print(doc.page_content)
  content_text.append(doc.page_content)
  print("-"*30)
"\n\n".join(content_text)
'''
context_text = "\n\n".join([doc.page_content for doc in docs_result])
context_text

In [ ]:
texts = "카카오는 인공지능 기술을 활용해 새로운"
answer = qa_chain.invoke({"context":context_text, "question":texts})

In [ ]:
answer['text']

In [ ]:
import torch
from transformers import BitsAndBytesConfig #양자화
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

In [ ]:
model_id = "kakaocorp/kanana-nano-2.1b-base"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    quantization_config = bnb_config,
    dtype="auto"
)

In [ ]:
model.is_loaded_in_4bit

In [ ]:
#pipline 설정
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=300,
    temperature=0.4)

In [ ]:
llm = HuggingFacePipeline(pipeline=pipe)

In [ ]:
prompt = ChatPromptTemplate([
    ("system","""
        너는 냉장고 제품 전문가야.
        아래 메뉴얼 내용을 참고해서 질문에 답변해.
        {context}
    """),
    ("user","{question}")
])
qa_chain = LLMChain(llm=llm, prompt=prompt, verbose=True)

In [ ]:
texts = "카카오는 인공지능 기술을 활용해 새로운"
answer = qa_chain.invoke({"context":context_text, "question":texts})

In [ ]:
answer['text']

In [ ]:
!ls

In [ ]:
vectorstore.save_local("./faiss_index")

In [ ]:
!ls

In [ ]:
from google.colab import drive
drive.mount('/content/driver')

In [ ]:
model.save_pretrained("/content/driver/MyDrive/kanana_model")

In [ ]:
tokenizer.save_pretrained("/content/driver/MyDrive/kanana_model")
vectorstore.save_local("/content/driver/MyDrive/faiss_index")

### 모델 및 벡터db 로드

In [ ]:
model_id = "/content/driver/MyDrive/kanana_model"
db_id = "/content/driver/MyDrive/faiss_index"

In [ ]:
model_serving = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto", dtype="auto")
tokenizer_serving = AutoTokenizer.from_pretrained(model_id)

In [ ]:
model_serving.is_loaded_in_4bit

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name="BM-K/KoSimCSE-roberta")

In [ ]:
db_id
vct = FAISS.load_local(db_id, embeddings,
                       allow_dangerous_deserialization=True)

In [ ]:
#pipline 설정
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=300,
    temperature=0.4)

llm = HuggingFacePipeline(pipeline=pipe)

prompt = ChatPromptTemplate([
    ("system","""
        한국어 문서를 분석하는 AI agent야.
        아래 문서 내용을 보고 답해
        {context}
    """),
     ("user","{question}")
    ])
qa_chain = LLMChain(llm = llm, prompt=prompt, verbose=True)

In [ ]:
texts = "카카오는 인공지능 기술을 활용해 새로운"
docs_re = vct.similarity_search(texts, k=1)

In [ ]:
con_text = "\n\n".join([doc.page_content for doc in docs_re])
con_text

In [ ]:
awswer = qa_chain.invoke({"context":con_text, "question":texts})

In [ ]:
answer